# Stage 4 — Base LLM without fine-tuning (zero-shot and 6-shot)

`HuggingFaceTB/SmolLM3-3B`, used as is, classifies RePro **dev** reviews with two prompts
(zero-shot and 6-shot). This is the "before" for the QLoRA fine-tuning of Stage 5.
Decisions: D15 in `docs/decisions.md`.

Runs on **Google Colab** or **Kaggle**; the first code cell detects which one.

- **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- **Kaggle:** in *Settings*, set *Accelerator → GPU T4 x2* and turn *Internet* on, then
  *Save Version → Save & Run All*. The zip appears in the version's *Output* tab.

What this notebook does:

1. Clones the repository and rebuilds the data, checking it matches the committed summary.
2. Builds the prompts with the model's chat template (thinking mode off).
3. **Label scoring (main method):** for each review, the log-probability of each of the 3 labels
   as the answer; the highest wins. One forward pass per (review, label): 3 per review.
4. **Generation (secondary, dev only):** greedy answer of a few tokens, mapped to a label by
   `parse_label`; answers that name no label (or several) count as invalid.
5. Picks zero-shot or 6-shot on dev, compares it with TF-IDF and BERTimbau (paired bootstrap),
   and predicts the **test** set with the chosen prompt, without test metrics (decision D12).
6. Packs the report and the prediction files into `smollm3_base_results.zip`.

Run time is not known yet; it is measured and saved in the report.

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

# Where are we? Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab ships the google.colab package.
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM = "kaggle"
    WORK_DIR = "/tmp"                # scratch space: the clone and data are not kept as output
    OUTPUT_DIR = "/kaggle/working"   # only files here are saved with the notebook version
elif importlib.util.find_spec("google.colab") is not None:
    PLATFORM = "colab"
    WORK_DIR = OUTPUT_DIR = "/content"
else:
    raise RuntimeError("Run this notebook on Google Colab or Kaggle")
print("platform:", PLATFORM)

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2"
)
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)

REPO_URL = "https://github.com/PedroRicci12/ptbr-review-lora.git"
REPO_DIR = f"{WORK_DIR}/ptbr-review-lora"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository commit:", REPO_COMMIT)

## 1. Data

Same script as on the laptop. If the counts differ from the committed
`reports/data_summary.json`, the cell stops.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_data.py"],
    check=True, stdout=subprocess.DEVNULL, env={**os.environ, "PYTHONPATH": "src"},
)
diff = subprocess.run(
    ["git", "diff", "--exit-code", "reports/data_summary.json"], capture_output=True, text=True
)
assert diff.returncode == 0, "Prepared data differs from the repository:\n" + diff.stdout
print("data_summary.json identical to the repository")

## 2. Configuration (fixed, not tuned)

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
MODEL_REVISION = "a07cc9a04f16550a088caea529712d1d335b0ac1"  # pinned; weights are safetensors

SEED = 42
USE_TITLE = True             # same input as TF-IDF and BERTimbau
FEW_SHOT_PER_CLASS = 2       # 6-shot (D15)
DTYPE = torch.float16        # the T4 has no bf16 support; numerical problems are checked below
SCORE_BATCH = 8              # reviews per forward pass (x3 labels = 24 sequences)
GEN_BATCH = 32
MAX_NEW_TOKENS = 8           # enough for one word; longer answers are cut and parsed anyway
N_RESAMPLES = 1000
REFERENCES = ("tfidf_lr__title_text__balanced", "bertimbau")

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from pathlib import Path

import pandas as pd

from ptbr_review_lora.data import LABELS, model_input
from ptbr_review_lora.prompts import build_messages, sample_few_shot

pool = pd.read_parquet("data/processed/b2w_pool.parquet")
dev = pd.read_parquet("data/processed/repro_dev.parquet")
test = pd.read_parquet("data/processed/repro_test.parquet")

dev_texts = model_input(dev, USE_TITLE).tolist()
test_texts = model_input(test, USE_TITLE).tolist()
y_dev = dev["label"].tolist()

# Few-shot examples come from the training pool (star labels), never from dev or test.
few_shot = sample_few_shot(pool, per_class=FEW_SHOT_PER_CLASS, seed=SEED, use_title=USE_TITLE)
SETTINGS = {
    "zero_shot": [],
    "6_shot": list(zip(few_shot["text"], few_shot["label"])),
}
print("few-shot pool rows:", few_shot.index.tolist(), few_shot["label"].tolist())

## 3. Model and chat template

- SmolLM3's chat template writes **today's date** into the system prompt by default, so the same
  review would get a different prompt on another day. The `/system_override` flag (read by the
  template) keeps only our instruction, which makes the prompt reproducible.
- `enable_thinking=False` adds an empty `<think></think>` block, so the answer comes right away.
- The template already contains the special tokens, so texts are tokenized with
  `add_special_tokens=False` (otherwise some tokens could appear twice).

In [ ]:
import transformers
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer

assert Version(transformers.__version__) >= Version("4.53"), "SmolLM3 needs transformers >= 4.53"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE).cuda().eval()
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id


def render(review, examples):
    """Prompt text for one review, ending where the assistant's answer starts."""
    messages = build_messages(review, examples)
    messages[0]["content"] += "\n/system_override"  # SmolLM3-specific: no date in the prompt
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )


def encode_prompts(texts, examples):
    return [tokenizer(render(t, examples), add_special_tokens=False)["input_ids"] for t in texts]


# Show the exact prompt format with a placeholder instead of a real review.
print(render("<AVALIAÇÃO>", SETTINGS["zero_shot"]))
assert "Today Date" not in render("x", []), "the date is still in the prompt"

LABEL_IDS = [tokenizer(label, add_special_tokens=False)["input_ids"] for label in LABELS]
print("label tokens:", {label: tokenizer.convert_ids_to_tokens(ids) for label, ids in zip(LABELS, LABEL_IDS)})

prompt_lengths = {}
for name, examples in SETTINGS.items():
    lengths = np.array([len(ids) for ids in encode_prompts(dev_texts, examples)])
    prompt_lengths[name] = {p: int(np.percentile(lengths, p)) for p in (50, 95, 100)}
print("prompt length in tokens (dev):", prompt_lengths)

## 4. Label scoring

For each review, 3 sequences: prompt + `negativo`, prompt + `neutro`, prompt + `positivo`.
The score of a label is the sum of the log-probabilities of its tokens, each predicted from
everything before it. The highest score wins, so every prediction is a valid label.

- Sequences are right-padded; with causal attention, padding at the end does not change the
  real tokens, and it is masked anyway.
- Only the positions that predict label tokens go through the output layer: the vocabulary has
  ~128k tokens, and computing logits for every position of 24 sequences would waste GBs of memory.
- The first check compares this shortcut with the plain `model(...).logits` on a few reviews.

In [ ]:
import time

from ptbr_review_lora.prompts import choose_label

decoder = model.get_decoder()           # transformer layers + final norm, without the output layer
lm_head = model.get_output_embeddings()  # hidden state -> vocabulary logits


@torch.inference_mode()
def score_batch(prompts):
    """Log-probability of each label as the answer, for a list of tokenized prompts -> (n, 3)."""
    seqs = [prompt + label_ids for prompt in prompts for label_ids in LABEL_IDS]
    input_ids = torch.full((len(seqs), max(map(len, seqs))), PAD_ID)
    attention_mask = torch.zeros_like(input_ids)
    rows, positions, targets = [], [], []
    for s, seq in enumerate(seqs):
        input_ids[s, : len(seq)] = torch.tensor(seq)
        attention_mask[s, : len(seq)] = 1
        prompt_len = len(prompts[s // len(LABELS)])
        for k, token in enumerate(LABEL_IDS[s % len(LABELS)]):
            rows.append(s)
            positions.append(prompt_len - 1 + k)  # the position *before* a token predicts it
            targets.append(token)

    hidden = decoder(input_ids=input_ids.cuda(), attention_mask=attention_mask.cuda()).last_hidden_state
    rows_t = torch.tensor(rows, device="cuda")
    log_probs = torch.log_softmax(lm_head(hidden[rows_t, torch.tensor(positions, device="cuda")]).float(), dim=-1)
    token_scores = log_probs[torch.arange(len(rows), device="cuda"), torch.tensor(targets, device="cuda")]
    scores = torch.zeros(len(seqs), device="cuda").index_add_(0, rows_t, token_scores)
    return scores.view(len(prompts), len(LABELS)).cpu().numpy()


@torch.inference_mode()
def score_plain(prompt):
    """Same scores computed the obvious way (one sequence at a time, full logits): for checking."""
    out = []
    for label_ids in LABEL_IDS:
        ids = torch.tensor([prompt + label_ids], device="cuda")
        log_probs = torch.log_softmax(model(input_ids=ids).logits[0].float(), dim=-1)
        out.append(sum(log_probs[len(prompt) - 1 + k, t].item() for k, t in enumerate(label_ids)))
    return np.array(out)


def score_all(prompts):
    """Scores for all prompts, batched by length (longest first, so memory errors appear early)."""
    order = np.argsort([-len(p) for p in prompts])
    scores = np.empty((len(prompts), len(LABELS)))
    for start in range(0, len(prompts), SCORE_BATCH):
        idx = order[start : start + SCORE_BATCH]
        scores[idx] = score_batch([prompts[i] for i in idx])
    return scores


check = encode_prompts(dev_texts[:4], SETTINGS["6_shot"])
fast, plain = score_batch(check), np.stack([score_plain(p) for p in check])
print("max |shortcut - plain|:", float(np.abs(fast - plain).max()))
assert np.allclose(fast, plain, atol=0.05), "the batched shortcut does not match the plain computation"
assert (fast.argmax(1) == plain.argmax(1)).all()

In [ ]:
from ptbr_review_lora.metrics import bootstrap_ci, compute_metrics, neutral_recall_by_group


def evaluate(preds):
    _, low, high = bootstrap_ci(y_dev, preds, n_resamples=N_RESAMPLES, seed=SEED)
    return {
        **compute_metrics(y_dev, preds),
        "macro_f1_ci95": [low, high],
        "neutral_recall_by_group": neutral_recall_by_group(dev["polarity_group"], preds),
    }


dev_preds, results, seconds = {}, {}, {}
for name, examples in SETTINGS.items():
    start = time.perf_counter()
    scores = score_all(encode_prompts(dev_texts, examples))
    seconds[f"score_dev__{name}"] = round(time.perf_counter() - start)
    n_bad = int((~np.isfinite(scores)).sum())
    assert n_bad == 0, f"{n_bad} non-finite scores: fp16 problem, try DTYPE = torch.float32"
    dev_preds[name] = [choose_label(dict(zip(LABELS, row))) for row in scores]
    results[name] = {"method": "label_scoring", **evaluate(dev_preds[name])}
    print(f"{name}: macro-F1 {results[name]['macro_f1']:.4f} "
          f"(95% CI {results[name]['macro_f1_ci95'][0]:.4f}-{results[name]['macro_f1_ci95'][1]:.4f}), "
          f"{seconds[f'score_dev__{name}']} s")

## 5. Generation (secondary, dev only)

The model writes its own answer (greedy, at most `MAX_NEW_TOKENS` tokens), and `parse_label`
maps it to a label. This measures whether the model follows the answer format, which label
scoring hides. Invalid answers count as errors in the metrics.

In [ ]:
from ptbr_review_lora.prompts import parse_label

INVALID = "invalido"


@torch.inference_mode()
def generate_all(prompts):
    tokenizer.padding_side = "left"  # generation continues from the end, so pad on the left
    order = np.argsort([-len(p) for p in prompts])
    answers = [None] * len(prompts)
    for start in range(0, len(prompts), GEN_BATCH):
        idx = order[start : start + GEN_BATCH]
        batch = tokenizer.pad({"input_ids": [prompts[i] for i in idx]}, return_tensors="pt").to("cuda")
        out = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=PAD_ID)
        new_tokens = out[:, batch["input_ids"].shape[1]:]
        for i, text in zip(idx, tokenizer.batch_decode(new_tokens, skip_special_tokens=True)):
            answers[i] = text
    return answers


answer_examples = {}
for name, examples in SETTINGS.items():
    start = time.perf_counter()
    answers = generate_all(encode_prompts(dev_texts, examples))
    seconds[f"generate_dev__{name}"] = round(time.perf_counter() - start)
    parsed = [parse_label(a) for a in answers]
    preds = [p if p is not None else INVALID for p in parsed]
    gen_name = f"{name}__generate"
    dev_preds[gen_name] = preds
    results[gen_name] = {
        "method": "generation",
        "invalid_rate": float(np.mean([p is None for p in parsed])),
        "agreement_with_label_scoring": float(np.mean([a == b for a, b in zip(preds, dev_preds[name])])),
        **evaluate(preds),
    }
    # Most frequent raw answers (just the model's words, no review text).
    answer_examples[gen_name] = pd.Series(answers).str.strip().value_counts().head(10).to_dict()
    print(f"{gen_name}: macro-F1 {results[gen_name]['macro_f1']:.4f}, "
          f"invalid {results[gen_name]['invalid_rate']:.1%}, {seconds[f'generate_dev__{name}']} s")
    print("  most frequent answers:", answer_examples[gen_name])

## 6. Choice on dev and comparison with the baselines

The prompt (zero-shot or 6-shot) is chosen by dev macro-F1 with label scoring, like the TF-IDF
configuration was. All models are compared on the same dev reviews (paired bootstrap, D11).

In [ ]:
from ptbr_review_lora.metrics import paired_bootstrap
from ptbr_review_lora.predictions import load_predictions, save_predictions

BEST = max(SETTINGS, key=lambda name: results[name]["macro_f1"])
print("chosen prompt:", BEST)

comparisons = {"6_shot vs zero_shot": paired_bootstrap(
    y_dev, dev_preds["zero_shot"], dev_preds["6_shot"], n_resamples=N_RESAMPLES, seed=SEED
)}
for ref in REFERENCES:
    ref_preds = load_predictions(Path(f"predictions/dev/{ref}.csv"), len(dev))
    comparisons[f"smollm3_base__{BEST} vs {ref}"] = paired_bootstrap(
        y_dev, ref_preds, dev_preds[BEST], n_resamples=N_RESAMPLES, seed=SEED
    )
for name, c in comparisons.items():
    print(f"{name}: diff {c['diff']:+.4f} (95% CI {c['low']:+.4f} to {c['high']:+.4f}), "
          f"p_b_not_better {c['p_b_not_better']:.3f}")

## 7. Test predictions with the chosen prompt (no metrics)

Decision D12: the test set is scored once, in Stage 6, for all frozen models together.

In [ ]:
start = time.perf_counter()
test_scores = score_all(encode_prompts(test_texts, SETTINGS[BEST]))
seconds[f"score_test__{BEST}"] = round(time.perf_counter() - start)
assert np.isfinite(test_scores).all()
test_preds = [choose_label(dict(zip(LABELS, row))) for row in test_scores]

for name, preds in dev_preds.items():
    save_predictions(Path(f"predictions/dev/smollm3_base__{name}.csv"), preds)
save_predictions(Path("predictions/test/smollm3_base.csv"), test_preds)
print(len(test_preds), "test predictions,", seconds[f"score_test__{BEST}"], "s")

## 8. Report and download

In [ ]:
import json
import platform
import zipfile

import sklearn

report = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "repo_commit": REPO_COMMIT,
    "environment": {
        "platform": PLATFORM,
        "gpu": GPU_NAME,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
    },
    "config": {
        "dtype": str(DTYPE), "use_title": USE_TITLE, "few_shot_per_class": FEW_SHOT_PER_CLASS,
        "max_new_tokens": MAX_NEW_TOKENS, "thinking": False, "seed": SEED,
    },
    "few_shot_pool_rows": few_shot.index.tolist(),
    "few_shot_labels": few_shot["label"].tolist(),
    "label_tokens": {label: len(ids) for label, ids in zip(LABELS, LABEL_IDS)},
    "prompt_length_tokens": prompt_lengths,
    "seconds": seconds,
    "split": "repro_dev",
    "n": len(dev),
    "n_resamples": N_RESAMPLES,
    "chosen_prompt": BEST,
    "results": results,
    "most_frequent_answers": answer_examples,
    "paired_comparisons": comparisons,
}
Path("reports/smollm3_base_dev.json").write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

files_to_pack = ["reports/smollm3_base_dev.json", "predictions/test/smollm3_base.csv"]
files_to_pack += [f"predictions/dev/smollm3_base__{name}.csv" for name in dev_preds]
ZIP_PATH = f"{OUTPUT_DIR}/smollm3_base_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_pack:
        zf.write(path)  # relative paths, so the files unpack into the same folders of the repository
print("saved", ZIP_PATH)

if PLATFORM == "colab":
    from google.colab import files

    files.download(ZIP_PATH)
else:
    print("Kaggle: download it from the Output tab of this notebook version")

## Next step

Hand `smollm3_base_results.zip` over: its files go into the repository at the same paths
(`reports/` and `predictions/`).